# H-009 Baseline Experiment

**WP-4b2** | H-009: Multi-Timeframe Trend Alignment | 2026-04-17

Primary timeframe: 30m | Confirmation: 1h, 4h | Free parameters: 5 | Trial ID: `7292282d-abcb-462c-8d4a-c8572ff529d7`

Evaluation note: no separate IS/OOS split convention exists in repo truth. The baseline uses the full registered evaluation range for both `in_sample_sharpe` and `oos_sharpe`, matching the current runbook baseline pattern.

Gate 2 semantics note: this notebook reports Gate 2 according to current `lib/evaluation.py` repo truth. The displayed legacy `GateReport` wording may read `DSR > 1.0 at 95% CI`, but the evaluator currently checks whether `dsr_val` exceeds 0.95 against its expected maximum-SR benchmark from the registered trial count.

Causal alignment note: 1h and 4h trend states are computed at their own bar closes. They are aligned to the 30m primary index by unioning indexes and forward-filling — a coarser bar closing at time T is not visible to any 30m bar before T. A spot-check confirms alignment correctness (see output below).

In [ ]:
from pathlib import Path
import json
import hashlib
import math
import pandas as pd
import numpy as np
from lib.backtest import run_backtest
from lib.evaluation import check_promotion_gates, cost_drag, hit_rate, max_drawdown, profit_factor, sharpe

HYPOTHESIS_ID = "H-009"
TRIAL_ID = "7292282d-abcb-462c-8d4a-c8572ff529d7"
TRIAL_COUNT_BEFORE_APPEND = 9
DATASET_IDS = [
    "binance_um_perp_BTCUSDT_30m_2020-01-01_2026-04-11_2026-04-12T1655Z",
    "binance_um_perp_BTCUSDT_1h_2020-01-01_2026-04-11_2026-04-12T1655Z",
    "binance_um_perp_BTCUSDT_4h_2020-01-01_2026-04-11_2026-04-12T1655Z",
    "binance_um_perp_BTCUSDT_funding_8h_2019-09-10_2026-04-12_2026-04-12T1655Z",
]
PARAMS = {"trigger_lookback_30m": 20, "trend_lookback_1h": 48, "trend_lookback_4h": 30, "atr_multiplier": 2.0, "time_stop_bars": 24}
COSTS = {"fee_bps_taker": 4.0, "fee_bps_maker": 2.0, "slippage_bps": 2.0, "initial_equity": 10000.0}

DATA_ROOT = Path("data/processed")
OHLCV_30M_PATH = DATA_ROOT / "binance_um_perp_BTCUSDT_30m_2020-01-01_2026-04-11_2026-04-12T1655Z.parquet"
OHLCV_1H_PATH  = DATA_ROOT / "binance_um_perp_BTCUSDT_1h_2020-01-01_2026-04-11_2026-04-12T1655Z.parquet"
OHLCV_4H_PATH  = DATA_ROOT / "binance_um_perp_BTCUSDT_4h_2020-01-01_2026-04-11_2026-04-12T1655Z.parquet"
FUNDING_PATH   = DATA_ROOT / "binance_um_perp_BTCUSDT_funding_8h_2019-09-10_2026-04-12_2026-04-12T1655Z.parquet"

def _sha256_of_dict(d):
    return hashlib.sha256(json.dumps(d, sort_keys=True, default=str).encode()).hexdigest()

def load_ohlcv(path):
    df = pd.read_parquet(path).copy()
    df.index = pd.to_datetime(df["close_time_ms"].astype("int64"), unit="ms", utc=True)
    return df[["open", "high", "low", "close", "volume"]].astype(float).sort_index()

def load_funding():
    df = pd.read_parquet(FUNDING_PATH).copy()
    df.index = pd.to_datetime(df["fundingTime"].astype("int64"), unit="ms", utc=True)
    return df.rename(columns={"fundingRate": "funding_rate"})[["funding_rate"]].astype(float).sort_index()

def filter_applicable_funding(signal, ohlcv, funding):
    common_idx = signal.index.intersection(ohlcv.index)
    sig = signal.reindex(common_idx).fillna(0).astype(int)
    bars = list(common_idx)
    position = 0
    event_ts = []
    funding_idx = funding.index
    for i, bar_ts in enumerate(bars):
        if i > 0 and position != 0:
            prev_bar_ts = bars[i - 1]
            matches = funding_idx[(funding_idx > prev_bar_ts) & (funding_idx <= bar_ts)]
            event_ts.extend(matches)
        if i > 0:
            desired = int(sig.iloc[i - 1])
            if desired != position:
                position = desired
    if not event_ts:
        return funding.iloc[0:0].copy()
    return funding.loc[pd.DatetimeIndex(sorted(set(event_ts)))]

def run_candidate(signal, ohlcv, funding):
    applicable = filter_applicable_funding(signal, ohlcv, funding)
    result = run_backtest(signal, ohlcv, applicable, **COSTS, execution="next_bar_open", stream="klines", interval="30m")
    return result, len(applicable)

def summarize(result):
    returns = result.equity_curve.pct_change().dropna()
    mdd, peak, trough = max_drawdown(result.equity_curve)
    cd = cost_drag(result)
    return {"final_equity": float(result.equity_curve.iloc[-1]), "total_pnl": float(result.equity_curve.iloc[-1] - result.equity_curve.iloc[0]), "sharpe": float(sharpe(returns, "30m")), "max_drawdown": float(mdd), "max_drawdown_peak": str(peak) if peak is not None else None, "max_drawdown_trough": str(trough) if trough is not None else None, "n_obs": int(len(returns)), "n_trades": int(len(result.trades_df)), "hit_rate": float(hit_rate(result.trades_df)), "profit_factor": float(profit_factor(result.trades_df)), "fees_paid": float(result.fees_paid), "slippage_paid": float(result.slippage_paid), "funding_paid": float(result.funding_paid), "gross_pnl": float(cd["gross_pnl"]), "cost_drag_fraction": float(cd["cost_drag_fraction"]) if math.isfinite(cd["cost_drag_fraction"]) else None, "skipped_bars": int(result.skipped_bars)}

def side_summary(result):
    if result.trades_df.empty:
        return []
    return [{"direction": int(d), "count": int(len(g)), "net_pnl_sum": float(g["net_pnl"].sum()), "net_pnl_mean": float(g["net_pnl"].mean())} for d, g in result.trades_df.groupby("direction")]

def sparkline(series, width=72):
    vals = series.dropna()
    if vals.empty: return ""
    if len(vals) > width: vals = vals.iloc[::max(1, len(vals) // width)].iloc[:width]
    chars = "._-~=+*#%@"
    lo, hi = float(vals.min()), float(vals.max())
    if hi == lo: return chars[0] * len(vals)
    return "".join(chars[min(len(chars) - 1, int((float(v) - lo) / (hi - lo) * (len(chars) - 1)))] for v in vals)

def drawdown_series(equity):
    return (equity - equity.cummax()) / equity.cummax()

def compute_atr(ohlcv, period=14):
    high = ohlcv["high"]; low = ohlcv["low"]; close = ohlcv["close"]
    tr = pd.concat([high - low, (high - close.shift(1)).abs(), (low - close.shift(1)).abs()], axis=1).max(axis=1)
    return tr.rolling(period, min_periods=period).mean()

def align_coarser_to_primary(coarser_series, primary_index):
    combined_idx = primary_index.union(coarser_series.index).sort_values()
    return coarser_series.reindex(combined_idx).ffill().reindex(primary_index)

def h009_features(ohlcv_30m, ohlcv_1h, ohlcv_4h, trigger_lookback_30m=20, trend_lookback_1h=48, trend_lookback_4h=30):
    tl = int(round(trigger_lookback_30m))
    trigger_high = ohlcv_30m["high"].rolling(tl, min_periods=tl).max().shift(1)
    trigger_low  = ohlcv_30m["low"].rolling(tl, min_periods=tl).min().shift(1)
    atr_30m = compute_atr(ohlcv_30m, period=14)
    tl1 = int(round(trend_lookback_1h))
    ma_1h = ohlcv_1h["close"].rolling(tl1, min_periods=tl1).mean()
    trend_1h_raw = pd.Series(np.where(ohlcv_1h["close"] > ma_1h, 1.0, np.where(ohlcv_1h["close"] < ma_1h, -1.0, 0.0)), index=ohlcv_1h.index, name="trend_1h")
    trend_1h_raw[ma_1h.isna()] = np.nan
    tl4 = int(round(trend_lookback_4h))
    ma_4h = ohlcv_4h["close"].rolling(tl4, min_periods=tl4).mean()
    trend_4h_raw = pd.Series(np.where(ohlcv_4h["close"] > ma_4h, 1.0, np.where(ohlcv_4h["close"] < ma_4h, -1.0, 0.0)), index=ohlcv_4h.index, name="trend_4h")
    trend_4h_raw[ma_4h.isna()] = np.nan
    trend_1h_aligned = align_coarser_to_primary(trend_1h_raw, ohlcv_30m.index)
    trend_4h_aligned = align_coarser_to_primary(trend_4h_raw, ohlcv_30m.index)
    return pd.DataFrame({"trigger_high": trigger_high, "trigger_low": trigger_low, "atr_30m": atr_30m, "trend_1h": trend_1h_aligned, "trend_4h": trend_4h_aligned}, index=ohlcv_30m.index)

def h009_signal(ohlcv_30m, ohlcv_1h, ohlcv_4h, trigger_lookback_30m=20, trend_lookback_1h=48, trend_lookback_4h=30, atr_multiplier=2.0, time_stop_bars=24):
    features = h009_features(ohlcv_30m, ohlcv_1h, ohlcv_4h, trigger_lookback_30m, trend_lookback_1h, trend_lookback_4h)
    atr_mult = float(atr_multiplier); t_stop = int(round(time_stop_bars))
    signal = pd.Series(0.0, index=ohlcv_30m.index, name="signal")
    current = 0.0; entry_price = None; atr_at_entry = None; bars_held = 0
    for ts in ohlcv_30m.index:
        close = float(ohlcv_30m.loc[ts, "close"])
        th = features.loc[ts, "trigger_high"]; tl = features.loc[ts, "trigger_low"]
        atr = features.loc[ts, "atr_30m"]; t1 = features.loc[ts, "trend_1h"]; t4 = features.loc[ts, "trend_4h"]
        if any(pd.isna(x) for x in [th, tl, atr, t1, t4]):
            signal.loc[ts] = current
            if current != 0.0: bars_held += 1
            continue
        long_entry  = (close > th) and (t1 == 1.0) and (t4 == 1.0)
        short_entry = (close < tl) and (t1 == -1.0) and (t4 == -1.0)
        if current == 1.0:
            bars_held += 1
            trend_exit  = (t1 != 1.0) or (t4 != 1.0)
            atr_exit = (atr_at_entry is not None) and (close < entry_price - atr_at_entry * atr_mult)
            time_exit = bars_held >= t_stop
            if trend_exit or short_entry or atr_exit or time_exit:
                if short_entry and not (atr_exit or time_exit):
                    current = -1.0; entry_price = close; atr_at_entry = float(atr); bars_held = 0
                else:
                    current = 0.0; entry_price = None; atr_at_entry = None; bars_held = 0
        elif current == -1.0:
            bars_held += 1
            trend_exit  = (t1 != -1.0) or (t4 != -1.0)
            atr_exit = (atr_at_entry is not None) and (close > entry_price + atr_at_entry * atr_mult)
            time_exit = bars_held >= t_stop
            if trend_exit or long_entry or atr_exit or time_exit:
                if long_entry and not (atr_exit or time_exit):
                    current = 1.0; entry_price = close; atr_at_entry = float(atr); bars_held = 0
                else:
                    current = 0.0; entry_price = None; atr_at_entry = None; bars_held = 0
        else:
            if long_entry:
                current = 1.0; entry_price = close; atr_at_entry = float(atr); bars_held = 0
            elif short_entry:
                current = -1.0; entry_price = close; atr_at_entry = float(atr); bars_held = 0
        signal.loc[ts] = current
    return signal, features

def sensitivity_h009(ohlcv_30m, ohlcv_1h, ohlcv_4h, funding, base_sharpe):
    base = PARAMS.copy()
    variants = [("trigger_lookback_30m_up25", {**base, "trigger_lookback_30m": int(round(20*1.25))}), ("trigger_lookback_30m_dn25", {**base, "trigger_lookback_30m": int(round(20*0.75))}), ("trend_lookback_1h_up25", {**base, "trend_lookback_1h": int(round(48*1.25))}), ("trend_lookback_1h_dn25", {**base, "trend_lookback_1h": int(round(48*0.75))}), ("trend_lookback_4h_up25", {**base, "trend_lookback_4h": int(round(30*1.25))}), ("trend_lookback_4h_dn25", {**base, "trend_lookback_4h": int(round(30*0.75))}), ("atr_multiplier_up25", {**base, "atr_multiplier": 2.5}), ("atr_multiplier_dn25", {**base, "atr_multiplier": 1.5}), ("time_stop_bars_up25", {**base, "time_stop_bars": int(round(24*1.25))}), ("time_stop_bars_dn25", {**base, "time_stop_bars": int(round(24*0.75))})]
    rows = []
    for name, params in variants:
        sig, _ = h009_signal(ohlcv_30m, ohlcv_1h, ohlcv_4h, **params)
        res, _ = run_candidate(sig, ohlcv_30m, funding)
        sr = summarize(res)["sharpe"]
        rows.append({"variant": name, "params": params, "sharpe": sr, "ratio_to_base": (sr / base_sharpe) if base_sharpe and base_sharpe != 0 else 0.0})
    return rows

ohlcv_30m = load_ohlcv(OHLCV_30M_PATH)
ohlcv_1h  = load_ohlcv(OHLCV_1H_PATH)
ohlcv_4h  = load_ohlcv(OHLCV_4H_PATH)
funding   = load_funding()

# Causal alignment spot check
test_4h_ts = ohlcv_4h.index[50]
ma_4h_ck = ohlcv_4h["close"].rolling(30, min_periods=30).mean()
trend_4h_ck = pd.Series(np.where(ohlcv_4h["close"] > ma_4h_ck, 1.0, -1.0), index=ohlcv_4h.index)
trend_4h_aligned_ck = align_coarser_to_primary(trend_4h_ck, ohlcv_30m.index)
pre_bars = ohlcv_30m.index[ohlcv_30m.index < test_4h_ts]
val_before = trend_4h_aligned_ck.loc[pre_bars[-1]] if len(pre_bars) > 0 else None
val_at = trend_4h_aligned_ck.loc[test_4h_ts] if test_4h_ts in trend_4h_aligned_ck.index else None
print(f"Causal alignment spot check: 4h bar={test_4h_ts}, 30m value just BEFORE={val_before}, 30m value AT 4h bar={val_at}")

signal, features = h009_signal(ohlcv_30m, ohlcv_1h, ohlcv_4h, **PARAMS)
signal_counts = {str(k): int(v) for k, v in signal.value_counts().sort_index().items()}
both_valid = features["trend_1h"].notna() & features["trend_4h"].notna()
both_agree = ((features["trend_1h"] == 1.0) & (features["trend_4h"] == 1.0)) | ((features["trend_1h"] == -1.0) & (features["trend_4h"] == -1.0))
filter_permissive_pct = float(both_agree.sum() / both_valid.sum() * 100) if both_valid.sum() > 0 else 0.0

result, funding_events = run_candidate(signal, ohlcv_30m, funding)
metrics = summarize(result)
sensitivity = sensitivity_h009(ohlcv_30m, ohlcv_1h, ohlcv_4h, funding, metrics["sharpe"])
sensitivity_min_ratio = min(r["ratio_to_base"] for r in sensitivity)

params_meta = {"n_free_params": 5, "sensitivity_min_ratio": sensitivity_min_ratio, "is_sharpe": metrics["sharpe"]}
gate_report = check_promotion_gates(result=result, trial_count=TRIAL_COUNT_BEFORE_APPEND, params_meta=params_meta, freq="30m")

config = {"hypothesis_id": HYPOTHESIS_ID, "primary_timeframe": "30m", "execution": "next_bar_open", "costs": COSTS, "params": PARAMS}
report = {"hypothesis_id": HYPOTHESIS_ID, "trial_id": TRIAL_ID, "trial_count_before_append": TRIAL_COUNT_BEFORE_APPEND, "dataset_ids": DATASET_IDS, "dataset_sha256": _sha256_of_dict({"ids": sorted(DATASET_IDS)}), "config_sha256": _sha256_of_dict(config), "params": PARAMS, "funding_events_applied": funding_events, "signal_counts": signal_counts, "filter_permissive_pct": filter_permissive_pct, "metrics": metrics, "sensitivity": sensitivity, "sensitivity_min_ratio": float(sensitivity_min_ratio), "side_summary": side_summary(result), "equity_sparkline": sparkline(result.equity_curve), "drawdown_sparkline": sparkline(drawdown_series(result.equity_curve)), "gate_report": gate_report.to_dict(), "verdict": "PASS" if gate_report.overall_pass else "NO-GO"}
print(json.dumps(report, indent=2, sort_keys=True))

Causal alignment spot check: 4h bar=2020-01-09 11:59:59.999000+00:00, 30m value just BEFORE=1.0, 30m value AT 4h bar=1.0
{
  "config_sha256": "40437885b4622f113842a0ef9e8dcebba2a89136da74305646a2dd1b9e401ab2",
  "dataset_sha256": "694e5b466fc94ec6388863d115301cbccdb991eab7dc0c0baafbc78edbc3e2ed",
  "filter_permissive_pct": 78.04, "funding_events_applied": 2151,
  "gate_report": {
    "gates": [
      {"gate_id": 1, "gate_name": "OOS Sharpe >= 1.0", "passed": false, "threshold": 1.0, "value": -0.1638},
      {"gate_id": 2, "gate_name": "DSR > 1.0 at 95% CI", "note": "P(SR>1.0)=0.0000; SR benchmark=0.4809", "passed": false, "threshold": 1.0, "value": 0.0},
      {"gate_id": 3, "gate_name": "Per-fold max DD <= 20%", "passed": false, "threshold": 0.2, "value": 9.2516},
      {"gate_id": 4, "gate_name": "Parameter sensitivity min_ratio >= 0.70", "passed": false, "threshold": 0.7, "value": -5.3017},
      {"gate_id": 5, "gate_name": "Cost drag <= 50% of gross PnL", "passed": false, "threshol

## Verdict

**NO-GO** — 2 of 8 gates passed (Gates 6, 7).

### Key Observations

| Metric | H-009 | H-006 (reference) |
|--------|-------|-------------------|
| Sharpe (full-period) | -0.1638 | -0.3512 |
| Max Drawdown | 925.16% | 612.08% |
| Total Trades | 2,148 | 926 |
| Fees Paid | $86,620 | $36,089 |
| Cost Drag | 864.96% | -194.29% (mech.) |
| Gross PnL | +$15,592 | -$27,929 |

Filter permissivity: **78.0%** of aligned 30m bars had both 1h and 4h trend filters in agreement. This is extremely high — the multi-TF filter allowed entry in nearly 4 out of every 5 bars, providing almost no selectivity over single-TF approaches. H-009 produced 2148 trades (2.3x H-006's 926), confirming the filter was insufficient to control turnover.

Unlike H-006, gross PnL was positive (+$15,592), but fees ($86,620) and slippage ($43,310) destroyed the account. Multi-TF alignment did produce slightly positive raw returns, but the alignment filters were too permissive to reduce cost drag.

Gates failed: Sharpe (G1), DSR (G2), Max DD (G3), Sensitivity (G4), Cost Drag (G5), Walk-Forward Efficiency (G8).

Causal alignment verified: spot check confirmed 4h bar closing at T is not visible on any 30m bar before T.

Decision record: [DEC-0009](../../../decisions/DEC-0009.md)